# Agentic RAG 

This notebook demonstrates the Agentic RAG logic exactly as it's implemented in your `backend` FastAPI service. It connects to your existing Pinecone vector store, local BM25 index, and Opentyphoon LLM.

You can use this notebook to rapidly iterate, test the system prompt, experiment with the agent's reasoning process, and debug issues without needing to spin up the API.

## 1. Environment Setup
We need to add the `backend` directory to the Python path so we can import modules directly from `src`. We also load the `.env` file to configure Pinecone and Opentyphoon API keys.

In [ ]:
import sys
import os

# Add backend directory to Python path to import from src
backend_path = os.path.abspath(os.path.join(os.getcwd(), '..', 'backend'))
if backend_path not in sys.path:
    sys.path.append(backend_path)

# Load environment variables
from dotenv import load_dotenv
load_dotenv(os.path.join(backend_path, '.env'))

print("Backend path added:", backend_path)

## 2. Configuration & Model Initialization
Load the backend's `config.yaml` to configure model names, index names, and local paths.

In [ ]:
from src.utils.config import load_config
from src.rag.retrieval import get_embedding_model, get_reranker, load_hybrid_store, HybridRetriever
import logging

logging.basicConfig(level=logging.INFO)

# Load configuration from backend/config.yaml
config = load_config()

# 1. Load Embedding Model
embedding_model = get_embedding_model(
    config["embedding"]["model_name"],
    config["embedding"]["device"],
)

# 2. Load Pinecone + BM25 stores
persist_dir_path = os.path.join(backend_path, config["vector_db"]["persist_directory"])
vectorstore, bm25_retriever = load_hybrid_store(
    embedding_model=embedding_model,
    persist_dir=persist_dir_path,
    index_name=config["vector_db"]["index_name"],
)

# 3. Load Reranker
reranker = get_reranker()

# 4. Initialize HybridRetriever
retriever = HybridRetriever(vectorstore, bm25_retriever, reranker)
print("Hybrid Retriever initialized successfully.")

## 3. Agent Tools and System Prompt
Here we unwrap the logic found in `src.rag.generator.RAGAgent` so you can interact with it directly.
We define the `search_knowledge_base` tool and the strict `SystemMessage`.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage

@tool
def search_knowledge_base(search_query: str) -> str:
    """Search the enterprise knowledge base for relevant documents. 
    Use this to find context for the user's question. You can use it multiple times with different queries if needed.
    Provide specific, focused queries.
    """
    print(f"\n\033[94m🛠️ [Agent Tool Call] Searching knowledge base for: '{search_query}'\033[0m")
    docs = retriever.search(search_query, k=4, fetch_k=10)
    
    if not docs:
        return "No relevant documents found for this query. Try a different search strategy or broader keywords."
    
    formatted_docs = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        formatted_docs.append(f"--- Document {i} (Source: {os.path.basename(source)}) ---\n{doc.page_content}")
    
    return "\n\n".join(formatted_docs)

system_message = SystemMessage(content=(
    "คุณคือ AI ผู้ช่วยเชี่ยวชาญด้าน 'คู่มือสำนักทะเบียนและประมวลผล มหาวิทยาลัยเชียงใหม่' (CMU Registrar Assistant)\n"
    "หน้าที่หลักของคุณคือการตอบคำถามเกี่ยวกับกฎระเบียบ การลงทะเบียน และบริการต่างๆ ของสำนักทะเบียนและประมวลผล มหาวิทยาลัยเชียงใหม่ โดยใช้ข้อมูลจากระบบค้นหาเอกสาร (search_knowledge_base) เท่านั้น\n\n"
    "=== กฎที่ต้องปฏิบัติตามอย่างเคร่งครัด (STRICT RULES) ===\n"
    "1. ค้นหาข้อมูลก่อนตอบเสมอ: ใช้เครื่องมือ search_knowledge_base เพื่อดึงข้อมูลที่เกี่ยวข้อง หากข้อมูลไม่เพียงพอ ให้ค้นหาด้วยคำค้นใหม่ (Search again)\n"
    "2. ห้ามคิดเอง: ห้ามแต่งข้อมูลขึ้นมาเองเด็ดขาด ตอบเฉพาะสิ่งที่มีปรากฏในเอกสารที่ค้นพบเท่านั้น\n"
    "3. กรณีไม่พบข้อมูล: หากค้นหาแล้วไม่พบคำตอบในเอกสาร หรืออยู่นอกเหนือจากเอกสาร ให้ตอบว่า 'ขออภัย ไม่พบข้อมูลในคู่มือสำนักทะเบียนฯ' เท่านั้น ห้ามตอบด้วยข้อมูลอื่นที่คุณรู้เด็ดขาด\n"
    "4. การขอความชัดเจน (Clarification): หากคำถามจากผู้ใช้สั้นเกินไป กำกวม หรือไม่ชัดเจน (เช่น พิมพ์มาแค่คำเดียวว่า 'เกรด' หรือ 'ลา') ห้ามเดาความหมาย ให้ถามกลับอย่างสุภาพเพื่อขอรายละเอียดเพิ่มเติม\n"
    "5. อ้างอิงแหล่งที่มา: หากตอบคำถามได้ ให้สรุปคำตอบให้กระชับ เข้าใจง่าย และระบุชื่อเอกสารต้นทาง (Source) เสมอ\n"
    "6. กฎการอ้างอิง: ห้ามสมมติหรือสร้างชื่อเอกสารขึ้นมาเองเด็ดขาด (ZERO Fake Source) ชื่อ Source ที่อ้างอิงต้องมาจากฟิลด์ (Source: [ชื่อไฟล์]) ในผลลัพธ์ของ search_knowledge_base เท่านั้น\n"
    "=================================================="
))

## 4. Build the LangGraph Agent
Initializes the LLM (`ChatOpenAI` connected to Opentyphoon) and compiles the ReAct agent graph.

In [ ]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(
    model="typhoon-v2.5-30b-a3b-instruct",
    api_key=os.environ.get("TYPHOON_API_KEY"),
    base_url="https://api.opentyphoon.ai/v1",
    temperature=0.1,
    max_tokens=2048,
)

tools = [search_knowledge_base]
agent = create_react_agent(llm, tools, prompt=system_message)
print("Agent graph compiled successfully.")

## 5. Execution and Testing
Run the agent using LangGraph's streaming mode to visualize the 'thoughts' (tool calls) and the final generation.

In [ ]:
def run_agent(query: str):
    print(f"\033[1m👤 User Query:\033[0m {query}\n" + "="*60)
    
    # Stream the agent's actions
    for chunk in agent.stream(
        {"messages": [("user", query)]},
        {"recursion_limit": 10},
        stream_mode="updates"
    ):
        if 'agent' in chunk:
            # Print LLM thought or generation
            msg = chunk['agent']['messages'][0]
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tool_call in msg.tool_calls:
                    print(f"\033[95m🧠 Agent Thought:\033[0m I need to use '{tool_call['name']}' with args {tool_call['args']}")
            else:
                print(f"\n\033[92m🤖 Final Answer:\033[0m\n{msg.content}\n" + "="*60)
        elif 'tools' in chunk:
            # Print Tool result
            msg = chunk['tools']['messages'][0]
            print(f"\033[93m📄 Tool Result ({msg.name}):\033[0m\n[Retrieved {len(msg.content)} chars of context...]\n" + "-"*60)

# Test an ambiguous query (Testing Rule #4: Clarification)
run_agent("เกรด")

In [ ]:
# Test a direct knowledge base query
run_agent("ลาพักการศึกษาต้องทำอย่างไร?")

## 6. Pre-packaged Backend Agent
You can also use the exact class defined in `backend/src/rag/generator.py` to see the simplified final outputs.

In [ ]:
from src.rag.generator import RAGAgent

# Initialize the encapsulated agent
backend_agent = RAGAgent(retriever=retriever)

answer, docs, steps = backend_agent.generate("ถ้าได้เกรด F ต้องลงเรียนใหม่ไหม?")

print("\033[92mFinal Answer:\033[0m\n", answer)

print("\n\033[93mSources used:\033[0m")
for doc in docs:
    print(f"- {os.path.basename(doc.metadata.get('source', 'Unknown'))}")